In [1]:
import numpy as np
import pandas as pd
from omegaconf import OmegaConf
import os
from hydra import compose, initialize
import json
from sklearn.metrics import confusion_matrix
from pathlib import Path
from misc.probe_data import ExperimentData
import polars as pl
import pandas as pd

def load_hydra_config_with_params(model, datapack, probe, config_name):
    with initialize(version_base="1.1", config_path="../trillema-of-truth/configs"):
        cfg = compose(config_name=config_name, overrides=[f"model={model}", f"datapack={datapack}", f"probe={probe}"])
    OmegaConf.set_struct(cfg, False)  # Allow overriding
    trial_name = cfg.trial_name
    if cfg.probe['name'] == 'mean_diff':
        cfg.search = False
    if cfg.search:
        trial_name += "_search"
    trial_name += f'_task-{cfg.task}'
    cfg["trial_name"] = trial_name
    # if cfg["task"] == 2:
    #     cfg["probe"]["assume_known_positives"] = False
    cfg["output_dir"] = os.path.join(cfg.output_dir, trial_name)
    OmegaConf.set_struct(cfg, True)
    return OmegaConf.to_container(cfg, resolve=True)

In [2]:
models = ['qwen-2.5-14b',
          'qwen-2.5-7b',
          'mistral-7B-v0.3',
          'gemma-2-9b',
          'gemma-7b',
          'llama-3-8b',
          'llama-3.2-3b',
          '_qwen-2.5-14b',
          '_qwen-2.5-7b',
          '_mistral-7B-v0.3',
          '_gemma-2-9b',
          '_gemma-7b',
          '_llama-3.1-8b',
          '_llama-3-8b-med',
          '_llama-3.1-8b-bio',
          '_llama-3.2-3b']
    

datapacks = ['cities_loc', 'med_indications', 'defs']
probes = ['zero_shot', 'mean_diff', 'ttpd', 'spca', 'svm', 'sawmil']
tasks = {
    'mean_diff': 3,
    'ttpd': 3,
    'spca': 3,
    'sawmil': -1,
    'svm': -1
}
model_names = {
    'llama-3-8b': 'Llama-3-8B',
    # 'llama-3.1-8b': 'Llama-3.1-8B',
    'llama-3.2-3b': 'Llama-3.2-3B',
    'mistral-7B-v0.3': 'Mistral-7B-v0.3',
    'qwen-2.5-7b': 'Qwen-2.5-7B',
    'qwen-2.5-14b': 'Qwen-2.5-14B',
    'gemma-7b': 'Gemma-7B',
    'gemma-2-9b': 'Gemma-2-9B',
    '_llama-3.1-8b': 'Llama-3.1-8B-Instruct',
    '_llama-3.2-3b': 'Llama-3.2-3B-Instruct',
    '_mistral-7B-v0.3': 'Mistral-7B-Instruct-v0.3',
    '_qwen-2.5-7b': 'Qwen-2.5-7B-Instruct',
    '_qwen-2.5-14b': 'Qwen-2.5-14B-Instruct',
    '_gemma-7b': 'Gemma-7B-it',
    '_gemma-2-9b': 'Gemma-2-9B-it',
    '_llama-3.1-8b-bio': 'Bio-Medical-Llama-3-8B',
    '_llama-3-8b-med': 'Llama3-Med42-8B'
}
model_shortnames = {
    'llama-3-8b': 'Llama-3-8B',
    'llama-3.1-8b': 'Llama-3.1-8B',
    'llama-3.2-3b': 'Llama-3.2-3B',
    'mistral-7B-v0.3': 'Mistral-7B-v0.3',
    'qwen-2.5-7b': 'Qwen-2.5-7B',
    'qwen-2.5-14b': 'Qwen-2.5-14B',
    'gemma-7b': 'Gemma-7B',
    'gemma-2-9b': 'Gemma-2-9B',
    '_llama-3.1-8b': 'Llama-3.1-8B',
    '_llama-3.2-3b': 'Llama-3.2-3B',
    '_mistral-7B-v0.3': 'Mistral-7B-v0.3',
    '_qwen-2.5-7b': 'Qwen-2.5-7B',
    '_qwen-2.5-14b': 'Qwen-2.5-14B',
    '_gemma-7b': 'Gemma-7B',
    '_gemma-2-9b': 'Gemma-2-9B',
    '_llama-3.1-8b-bio': 'Bio-Medical-Llama',
    '_llama-3-8b-med': 'Llama3-Med42-8B'
}

model_types ={
    'llama-3-8b': 'default',
    'llama-3.1-8b': 'default',
    'llama-3.2-3b': 'default',
    'mistral-7B-v0.3': 'default',
    'qwen-2.5-7b': 'default',
    'qwen-2.5-14b': 'default',
    'gemma-7b': 'default',
    'gemma-2-9b': 'default',
    '_llama-3.1-8b': 'chat',
    '_llama-3.2-3b': 'chat',
    '_mistral-7B-v0.3': 'chat',
    '_qwen-2.5-7b': 'chat',
    '_qwen-2.5-14b': 'chat',
    '_gemma-7b': 'chat',
    '_gemma-2-9b': 'chat',
    '_llama-3.1-8b-bio': 'chat',
    '_llama-3-8b-med': 'chat',
}
dataset_names = {
    'cities_loc': 'City Locations',
    'med_indications': 'Medical Indications',
    'defs': 'Word Definitions',
}

condition_names = {
    'bag': 'Bag-Level',
    'instance': 'Instance-Level',
    # 'instance_tf': 'Instance-Level (TF Only)'
}

probe_order = ["zero_shot","mean_diff", "ttpd", "spca", "svm", "sawmil"]
probe_names = {
    'zero_shot': 'Zero-Shot',
    'mean_diff': 'MD+CP',
    'ttpd': 'TTPD+CP',
    'spca': 'sPCA+CP',
    'svm': 'SVM',
    'sawmil': 'sAwMIL'
}
datapack_order = ['cities_loc', 'med_indications', 'defs']  
model_order = list(model_names.keys())

WITH_SEARCH = True

save_dir = Path("outputs/figures/summaries")
save_dir.mkdir(parents=True, exist_ok=True)

## Performance Summaries

In [3]:
keys = [ 'conformal', 'mcc']
WITH_SEARCH = True
result = []
for model, m_name in model_names.items():
    for datapack in datapacks:
        for probe in probes:
            if probe == 'zero_shot':
                dir = Path('outputs/probes/zero_shot/') / model / datapack / 'metrics.json'
                values = json.load(open(dir, 'r'))[keys[0]][keys[1]]
                result.append((model, 'bag', datapack, probe, 'zero_shot', values[1], values[0], values[2], 0, 0))
                result.append((model, 'instance', datapack, probe, 'zero_shot', values[1], values[0], values[2], 0, 0))
                continue
            config = load_hydra_config_with_params(model=model, datapack=datapack, probe=probe, config_name='probe_training')

            experiment = "g_" + datapack            
            eD = ExperimentData(model_name=model, dataset_name=datapack, task=tasks[probe], probe_name=probe, with_search=WITH_SEARCH)
            for cond in ['bag', 'instance']:
                try:
                    if probe == 'svm':
                        key_set = [cond] + ['default', 'mcc']
                    else:
                        key_set = [cond] + keys
                    lid = eD.best_layer(keys=key_set, path=eD.base_path / experiment)
                    values = eD.read_metrics(layer_id=lid, keys=key_set, path=eD.base_path / experiment)
                    relative_depth = lid / eD.available_layers[-1]
                    result.append((model, cond, datapack, probe, experiment, values[1], values[0], values[2], lid, relative_depth))
                except Exception as ex:
                    print(f"Error for {model}, {datapack}, {probe}, {experiment}: {ex}")

In [5]:
df = pd.DataFrame(result, columns=['model', 'condition', 'datapack', 'probe', 'experiment', 'CI_lower', 'stat', 'CI_upper', 'best_layer' ,'relative_depth'])
df['experiment'] = df['experiment'].str.replace('g_', '')

In [44]:
dfs = df.rename(columns={
    'model': 'Model',
    'condition': 'Eval. Setting',
    'datapack': 'Dataset',
    'probe': 'Probe',
    'CI_lower': 'CI .025',
    'stat': 'MCC',
    'CI_upper': 'CI .975',
    'best_layer': 'Best Layer',
    'relative_depth': 'Rel. Depth'
    
})
dfs['Model'] = dfs['Model'].astype(pd.CategoricalDtype(categories=list(model_names.keys()), ordered=True))
dfs['Dataset'] = dfs['Dataset'].astype(pd.CategoricalDtype(categories=datapack_order, ordered=True))
dfs = dfs.sort_values(by = ['Dataset', 'Model'])
dfs['Type'] = dfs['Model'].map(model_types)
dfs['Model'] = dfs['Model'].map(model_shortnames)
dfs['Dataset'] = dfs['Dataset'].map(dataset_names)
dfs['Probe'] = dfs['Probe'].map(probe_names)

###  FILTER HERE
dfs = dfs[(dfs['Probe'] == 'SVM') &\
         (dfs['Eval. Setting'] == 'instance')
        ]

# dfs['Best Layer'] = "-"
# dfs['Rel. Depth'] = "-"
# dfs["Eval. Setting"] = '-'


dfs = dfs.round(2)
dfs = dfs[['Model', 'Type', 'Eval. Setting',  'Probe', 'Dataset', 'CI .025', 'MCC', 'CI .975', 'Best Layer' ,'Rel. Depth']]
mask_strong = (
    (dfs['CI .025'] > 0) |
    (dfs['CI .975'] < 0)
)
dfs['MCC'] = [
    # val = original float, cond = True if CI excludes zero
    f"\\textbf{{{val:.2f}}}" if cond else f"{val:.2f}"
    for val, cond in zip(dfs['MCC'], mask_strong)
]
dfs['Probe'] = dfs['Probe'].apply(lambda x: f"\\texttt{{{x}}}")
print(dfs.to_latex(
    index=False,
    float_format="%.2f",
    escape=False,
    column_format='lcccc',
))

\begin{tabular}{lcccc}
\toprule
Model & Type & Eval. Setting & Probe & Dataset & CI .025 & MCC & CI .975 & Best Layer & Rel. Depth \\
\midrule
Llama-3-8B & default & instance & \texttt{SVM} & City Locations & 0.97 & \textbf{0.98} & 0.99 & 12 & 0.39 \\
Llama-3.2-3B & default & instance & \texttt{SVM} & City Locations & 0.95 & \textbf{0.96} & 0.97 & 9 & 0.33 \\
Mistral-7B-v0.3 & default & instance & \texttt{SVM} & City Locations & 0.96 & \textbf{0.97} & 0.98 & 12 & 0.39 \\
Qwen-2.5-7B & default & instance & \texttt{SVM} & City Locations & 0.96 & \textbf{0.97} & 0.98 & 18 & 0.67 \\
Qwen-2.5-14B & default & instance & \texttt{SVM} & City Locations & 0.97 & \textbf{0.97} & 0.98 & 24 & 0.51 \\
Gemma-7B & default & instance & \texttt{SVM} & City Locations & 0.97 & \textbf{0.98} & 0.99 & 15 & 0.56 \\
Gemma-2-9B & default & instance & \texttt{SVM} & City Locations & 0.97 & \textbf{0.98} & 0.99 & 21 & 0.51 \\
Llama-3.1-8B & chat & instance & \texttt{SVM} & City Locations & 0.97 & \textbf{0.98} &

#### Generalization

In [20]:
keys = [ 'conformal', 'mcc']
WITH_SEARCH = True
result = []
for model, m_name in model_names.items():
    for datapack in datapacks:
        for probe in probes:
            if probe == 'zero_shot':
                dir = Path('outputs/probes/zero_shot/') / model / datapack / 'metrics.json'
                values = json.load(open(dir, 'r'))[keys[0]][keys[1]]
                result.append((model, 'bag', datapack, probe, 'zero_shot', values[1], values[0], values[2], 0, 0))
                result.append((model, 'instance', datapack, probe, 'zero_shot', values[1], values[0], values[2], 0, 0))
                continue
            config = load_hydra_config_with_params(model=model, datapack=datapack, probe=probe, config_name='probe_training')

            for experiment in datapacks:
                experiment = "g_" + experiment            
                eD = ExperimentData(model_name=model, dataset_name=datapack, task=tasks[probe], probe_name=probe, with_search=WITH_SEARCH)
                for cond in ['bag', 'instance']:
                    try:
                        if probe == 'svm':
                            key_set = [cond] + ['default', 'mcc']
                        else:
                            key_set = [cond] + keys
                        lid = eD.best_layer(keys=key_set, path=eD.base_path / experiment)
                        values = eD.read_metrics(layer_id=lid, keys=key_set, path=eD.base_path / experiment)
                        relative_depth = lid / eD.available_layers[-1]
                        result.append((model, cond, datapack, probe, experiment.replace("g_", ""), values[1], values[0], values[2], lid, relative_depth))
                    except Exception as ex:
                        print(f"Error for {model}, {datapack}, {probe}, {experiment}: {ex}")

In [58]:
dfg = pd.DataFrame(result, columns=['model', 'condition', 'datapack', 'probe', 'experiment', 'CI_lower', 'stat', 'CI_upper', 'best_layer' ,'relative_depth'])
dfg = dfg[(dfg['probe'] != 'zero_shot') & (dfg['datapack'] != dfg['experiment'])]
dfg['datapack'] = dfg['datapack'].apply(lambda x: dataset_names[x])
dfg['condition'] = dfg['condition'].map(condition_names)
dfg['probe'] = dfg['probe'].map(probe_names)
dfg = dfg.groupby(['condition', 'probe', 'datapack']).agg({'stat': ['mean', 'sem']})
dfg = dfg.round(2)
dfg.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in dfg.columns]
dfg = dfg.reset_index()
dfg.columns = ['Eval. Setting', 'Probe', 'Dataset', 'MCC', 'Stand. Err.']
dfg['Probe'] = dfg['Probe'].apply(lambda x: f"\\texttt{{{x}}}")

print(dfg.to_latex(
    index=False,
    float_format="%.2f",
    escape=False,
    column_format='llrcc',
))

\begin{tabular}{llrcc}
\toprule
Eval. Setting & Probe & Dataset & MCC & Stand. Err. \\
\midrule
Bag-Level & \texttt{MD+CP} & City Locations & 0.02 & 0.00 \\
Bag-Level & \texttt{MD+CP} & Medical Indications & 0.03 & 0.01 \\
Bag-Level & \texttt{MD+CP} & Word Definitions & 0.04 & 0.02 \\
Bag-Level & \texttt{SVM} & City Locations & 0.24 & 0.03 \\
Bag-Level & \texttt{SVM} & Medical Indications & 0.37 & 0.04 \\
Bag-Level & \texttt{SVM} & Word Definitions & 0.40 & 0.04 \\
Bag-Level & \texttt{TTPD+CP} & City Locations & 0.11 & 0.03 \\
Bag-Level & \texttt{TTPD+CP} & Medical Indications & 0.09 & 0.03 \\
Bag-Level & \texttt{TTPD+CP} & Word Definitions & 0.13 & 0.04 \\
Bag-Level & \texttt{sAwMIL} & City Locations & 0.82 & 0.02 \\
Bag-Level & \texttt{sAwMIL} & Medical Indications & 0.88 & 0.01 \\
Bag-Level & \texttt{sAwMIL} & Word Definitions & 0.86 & 0.01 \\
Bag-Level & \texttt{sPCA+CP} & City Locations & 0.11 & 0.02 \\
Bag-Level & \texttt{sPCA+CP} & Medical Indications & 0.13 & 0.03 \\
Bag-Level 

## Confusion Matrices

In [29]:
import numpy as np
model = "_gemma-2-9b"
datapack = 'cities_loc'
probe = 'mean_diff'
WITH_SEARCH = True
cond = 'instance'
eD = ExperimentData(model_name=model, dataset_name=datapack, task=tasks[probe], probe_name=probe, with_search=WITH_SEARCH)
lid = eD.best_layer(keys=[cond, 'conformal', 'cm'], path=eD.base_path / f"g_{datapack}")
if probe in ['mean_diff', 'ttpd', 'spca']:
    keys = [ 'conformal', 'cm']
    cm = np.asarray(eD.read_metrics(layer_id=lid, keys=[cond] + keys, path=eD.base_path / f"g_{datapack}"))
    cm[:, [2, 3]] = cm[:, [3, 2]]
    order = [1, 0, 2, 3]
    cm = cm[order][:, order]
    cm = cm / cm.sum(axis=1, keepdims=True)
    cm = cm[:3]

cm

/var/folders/pk/3vzybg253k1d3n7qzxkts_2c0000gn/T/ipykernel_19288/163350137.py:15: RuntimeWarning: invalid value encountered in divide
  cm = cm / cm.sum(axis=1, keepdims=True)


array([[0.8912    , 0.0864    , 0.0224    , 0.        ],
       [0.05263158, 0.92569659, 0.02167183, 0.        ],
       [0.79109589, 0.18835616, 0.02054795, 0.        ]])

In [59]:
import joblib
keys = [ 'default', 'cm']
WITH_SEARCH = True
cond = 'bag'
cms = {}
probe = 'svm'
if probe in ['mean_diff', 'ttpd', 'spca']:
    keys = [ 'conformal', 'cm']


result = []
for model, m_name in model_names.items():
    for datapack in datapacks:
        config = load_hydra_config_with_params(model=model, datapack=datapack, probe=probe, config_name='probe_training')
        eD = ExperimentData(model_name=model, dataset_name=datapack, task=tasks[probe], probe_name=probe, with_search=WITH_SEARCH)

        try:
            key_set = [cond] + keys
            lid = eD.best_layer(keys=key_set, path=eD.base_path / f"g_{datapack}")
            cm = np.asarray(eD.read_metrics(layer_id=lid, keys=key_set, path=eD.base_path / f"g_{datapack}"))
            if probe in ['mean_diff', 'ttpd', 'spca']:
                cm[:, [2, 3]] = cm[:, [3, 2]]
                order = [1, 0, 2, 3]
                cm = cm[order][:, order]
                cm = cm[:3]
            else:
                # order = [1, 0, 2, 3]
                # cm = cm[order][:, order]
                cm = cm[:3]
                
            cm = cm / cm.sum(axis=1, keepdims=True)
            cms[(dataset_names[datapack], model_names[model])] = cm
        except Exception as ex:
            print(f"Error for {model}, {datapack}, {probe}, {experiment}: {ex}")


In [60]:
true_labels = ['T', 'F', 'N']
pred_labels = ['T', 'F', 'N', 'A']

frames = {}
for (ds,m), mat in cms.items():
    df = pd.DataFrame(mat, index=true_labels, columns=pred_labels)
    frames[(ds,m)] = df

# 1) Gather sorted model & dataset lists
_models   = sorted({m for (_, m) in frames.keys()})
_datasets = sorted({ds for (ds, _) in frames.keys()})

# 2) Build one giant “wide” DataFrame with a 2-level column index
big_df = pd.concat(
    {
        ds: pd.concat(
                [frames[(ds, m)] for m in _models],
                axis=1,
                keys=_models,
                names=['model','pred_label']
            )
        for ds in _datasets
    },
    axis=0,
    names=['dataset','true_label']
)

# 3) Stack into long form
flat = (
    big_df
      .stack(level=['model','pred_label'])    # bring model & pred_label into the index
      .reset_index(name='count')              # true_label, dataset, model, pred_label, count
)

# 4) Pivot to get a 2-level column index (true_label → pred_label)
compact_mi = flat.pivot_table(
    index   = ['model','dataset'],
    columns = ['true_label','pred_label'],
    values  = 'count',
    aggfunc = 'first'
)

# 5) Reindex rows & columns into exactly the order you want
desired_index = pd.MultiIndex.from_product(
    [_models, _datasets],
    names=['dataset','model']
)
desired_columns = pd.MultiIndex.from_product(
    [true_labels, pred_labels],
    names=['true_label','pred_label']
)

compact_mi = compact_mi.reindex(index=desired_index, columns=desired_columns)

# 6) Export straight to LaTeX (with booktabs + multirow support in your preamble)
latex_str = compact_mi.to_latex(
    caption      = "Normalized confusion matrices by true vs.\ predicted",
    label        = "tab:confusion",
    multicolumn  = True,
    multirow     = True,
    escape       = False,
    float_format="{:0.2f}".format,
)

print(latex_str)

\begin{table}
\caption{Normalized confusion matrices by true vs.\ predicted}
\label{tab:confusion}
\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 & true_label & \multicolumn{4}{r}{T} & \multicolumn{4}{r}{F} & \multicolumn{4}{r}{N} \\
 & pred_label & T & F & N & A & T & F & N & A & T & F & N & A \\
dataset & model &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{Bio-Medical-Llama-3-8B} & City Locations & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 \\
 & Medical Indications & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 \\
 & Word Definitions & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 \\
\cline{1-14}
\multirow[t]{3}{*}{Gemma-2-9B} & City Locations & 0.97 & 0.00 & 0.03 & 0.00 & 0.98 & 0.00 & 0.02 & 0.00 & 0.02 & 0.00 & 0.98 & 0.00 \\
 & Medical Indications & 1.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.00 & 0.00 & 0.00 & 0.82 & 0.00 & 0.18 & 0.00 \\
 & Word Definitions 

/var/folders/pk/3vzybg253k1d3n7qzxkts_2c0000gn/T/ipykernel_19288/1963263225.py:31: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=['model','pred_label'])    # bring model & pred_label into the index


In [147]:
scores

array([1., 1., 1., ..., 1., 1., 1.])